# TravelMind Notebook 3: Autonomous and Deterministic Orchestration
### IBS Agentic AI Practitioner Bootcamp | Day 6 | Advanced Strands

The last climb. Notebook 1 kept you in control; Notebook 2 handed steps to the model. Now we visit both extremes and then combine them.

**You will build:**
- **v7 Swarm**: peer agents with shared memory and no boss. The path emerges from the agents.
- **v8 Graph**: a deterministic, auditable structure you define. The model reasons inside nodes; you own the flow.
- **v9 Composition**: a compliance Graph with a creative Swarm living inside one node. Rails outside, creativity inside.

Same TravelMind, same three dials, same measurement harness.

## You are here, and the spectrum this notebook covers

```mermaid
flowchart LR
    subgraph Done [Notebooks 1-2]
        w[v1-v4 workflows] --> a[v5-v6 agentic]
    end
    subgraph Now [Notebook 3: the two ends, then both at once]
        s[v7 Swarm: model owns the path] --- g[v8 Graph: you own the structure]
        g --- c[v9 Composition: graph outside, swarm inside]
    end
    a --> s
```

Everything on Day 6 sits on one axis: **who controls the path?**

```mermaid
flowchart LR
    L[Swarm: least deterministic - peers hand off freely] --> M[Orchestrator / Evaluator: shared control] --> R[Graph: most deterministic - fixed structure, audit trail]
```

Swarm and Graph are the endpoints. Composition lets you place the creative part exactly where you can afford it, inside guardrails you control.

## How to run

**VS Code:** open the notebook, pick a Python 3.10+ kernel, ensure boto3 has AWS credentials with Bedrock access to Claude Haiku 4.5 in `us-east-1`, run top to bottom.

**Colab:** upload, run the install cell, set `os.environ["AWS_ACCESS_KEY_ID"]=...` (temporary role creds preferred) and `os.environ["AWS_REGION"]="us-east-1"`, run top to bottom.

Self-contained: it rebuilds the mock airline layer and harness, and adds three new operations (compensation, identity validation, audit logging) that the graph needs.

In [ ]:
%pip install -q strands-agents strands-agents-tools matplotlib

## Setup: model, base tools, harness

Same three cells as before. Then one extra cell adds the operations that an auditable rebooking flow needs.

In [ ]:
import os, time, json, asyncio, contextlib

# NOTE: do NOT use nest_asyncio here. Strands runs every Swarm/Graph invocation in a
# fresh thread via asyncio.run() (see strands._async.run_async), so it never needs a
# re-entrant loop. nest_asyncio.apply() monkeypatches asyncio globally, and on
# Python 3.14 its patched Task machinery leaves asyncio.current_task() == None inside
# the swarm's async generator, so `async with asyncio.timeout(...)` raises
# "RuntimeError: Timeout should be used inside a task". Leaving it out is the fix.

from strands import Agent, tool
from strands.models import BedrockModel

REGION   = os.environ.get("AWS_REGION", "us-east-1")
HAIKU_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # us. => cross-region inference profile

haiku    = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0.3)
haiku_t0 = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0)  # deterministic gate agents

print("Region:", REGION, "| Model:", HAIKU_ID)

In [2]:
# --- Base mock airline operations (from NB1/NB2) ---
_PNRS = {
    "JX48Q2": {
        "surname": "Rao", "passenger_id": "P-100294", "loyalty_tier": "Gold",
        "segments": [
            {"flight": "6E-317", "from": "BLR", "to": "DEL", "dep": "2026-07-05T08:10",
             "status": "CANCELLED", "fare_basis": "TA21R"},
            {"flight": "6E-512", "from": "DEL", "to": "BOM", "dep": "2026-07-05T13:40",
             "status": "ON TIME", "fare_basis": "TA21R"},
        ],
        "ancillaries": {"seat": "14C (paid)", "bags": 1},
    }
}
_FARE_RULES = {
    "TA21R": {"refundable": False, "change_fee": 3000, "currency": "INR",
              "notes": "Non-refundable on voluntary change. Airline-caused (involuntary) changes waive fees."},
}


@tool
def get_pnr(record_locator: str, surname: str) -> str:
    """Look up a booking by record locator and surname (identity check).

    Args:
        record_locator: 6-character PNR, e.g. 'JX48Q2'.
        surname: Passenger surname for verification.
    Returns:
        JSON string of the booking, or an error if not found / surname mismatch.
    """
    rec = _PNRS.get(record_locator.upper())
    if not rec or rec["surname"].lower() != surname.lower():
        return json.dumps({"error": "PNR not found or surname mismatch"})
    return json.dumps(rec)


@tool
def get_fare_rules(fare_basis: str) -> str:
    """Return fare rules (refundability, change fee) for a fare basis code.

    Args:
        fare_basis: Fare basis code, e.g. 'TA21R'.
    Returns:
        JSON string of fare rules.
    """
    return json.dumps(_FARE_RULES.get(fare_basis.upper(), {"error": "unknown fare basis"}))


@tool
def search_reaccommodation(origin: str, destination: str, after: str) -> str:
    """Find alternative flights to re-accommodate a passenger after a disruption.

    Args:
        origin: Origin airport code, e.g. 'BLR'.
        destination: Destination airport code, e.g. 'BOM'.
        after: ISO datetime; only flights departing after this are returned.
    Returns:
        JSON string list of candidate flights.
    """
    options = [
        {"flight": "6E-333", "from": origin, "to": destination, "dep": "2026-07-05T11:20", "seats": 4},
        {"flight": "AI-809", "from": origin, "to": destination, "dep": "2026-07-05T15:05", "seats": 9},
    ]
    return json.dumps(options)


@tool
def get_loyalty(passenger_id: str) -> str:
    """Return loyalty tier and benefits for a passenger.

    Args:
        passenger_id: Internal passenger id, e.g. 'P-100294'.
    Returns:
        JSON string of tier and benefits.
    """
    benefits = {"Gold": ["priority rebooking", "waived change fee on same-day", "lounge access"]}
    return json.dumps({"passenger_id": passenger_id, "tier": "Gold", "benefits": benefits["Gold"]})


@tool
def check_refund_eligibility(record_locator: str, reason: str) -> str:
    """Assess refund eligibility from the booking's fare rules and the stated reason.

    Args:
        record_locator: PNR.
        reason: Free-text reason, e.g. 'flight cancelled by airline'.
    Returns:
        JSON string with an eligibility signal and the controlling rule.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"error": "PNR not found"})
    fb = rec["segments"][0]["fare_basis"]
    rules = _FARE_RULES.get(fb, {})
    airline_caused = any(k in reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    eligible = airline_caused or rules.get("refundable", False)
    return json.dumps({"eligible": eligible, "airline_caused": airline_caused, "rule": rules.get("notes", "")})

print("Base tools ready.")

Base tools ready.


In [3]:
# --- Measurement harness (from NB2), reads single-agent and multi-agent usage ---
PRICES  = {"haiku": (1.00, 5.00), "sonnet": (3.00, 15.00)}   # illustrative; VERIFY before quoting
_LEDGER = []
RESULTS = {}


def usage_of(result) -> dict:
    u = getattr(getattr(result, "metrics", None), "accumulated_usage", None)
    if u is None:
        u = getattr(result, "accumulated_usage", None) or {}
    def g(*keys):
        for k in keys:
            if k in u:
                return u[k]
        return 0
    return {"input": g("inputTokens", "input_tokens"), "output": g("outputTokens", "output_tokens")}


def _nodes_run(result):
    """Ordered node ids that executed (Graph .execution_order or Swarm .node_history)."""
    seq = getattr(result, "execution_order", None) or getattr(result, "node_history", None) or []
    out = []
    for n in seq:
        out.append(getattr(n, "node_id", None) or getattr(n, "id", None) or str(n))
    return out


def final_text(result):
    ids = _nodes_run(result)
    if ids:
        try:
            return str(result.results[ids[-1]].result)
        except Exception:
            pass
    return str(result)


def _record(res, tier="haiku"):
    _LEDGER.append((tier, usage_of(res), 1))


def metered(agent, prompt, tier="haiku"):
    res = agent(prompt); _record(res, tier); return str(res)


def metered_multi(orchestrator, prompt, tier="haiku"):
    res = orchestrator(prompt)
    ncalls = max(1, len(_nodes_run(res)))
    _LEDGER.append((tier, usage_of(res), ncalls))
    return final_text(res), res


@contextlib.contextmanager
def meter(label):
    _LEDGER.clear()
    t0 = time.time()
    try:
        yield
    finally:
        latency = time.time() - t0
        cost = tin = tout = ncalls = 0
        for tier, u, c in _LEDGER:
            pin, pout = PRICES[tier]
            cost += u["input"] / 1e6 * pin + u["output"] / 1e6 * pout
            tin += u["input"]; tout += u["output"]; ncalls += c
        RESULTS[label] = {"latency_s": round(latency, 2), "input_tokens": tin,
                          "output_tokens": tout, "calls": ncalls, "cost_usd": round(cost, 6)}
        r = RESULTS[label]
        print(f"[{label}]  calls/nodes={r['calls']}  latency={r['latency_s']}s  "
              f"tokens={tin}in+{tout}out  cost=${r['cost_usd']:.6f}")

### New operations for an auditable flow

Three deterministic operations the rebooking graph needs. Two are **hard business rules** (identity, policy) that read ground-truth data, not model text. One is an **audit log** so every decision leaves a record. These are plain Python; no model guesses here.

In [4]:
# --- Deterministic operations: identity, compensation, policy gate, audit ---
AUDIT_LOG = []  # append-only compliance record


@tool
def validate_identity(record_locator: str, surname: str) -> str:
    """Hard identity check against ground-truth booking data.

    Args:
        record_locator: PNR, e.g. 'JX48Q2'.
        surname: Passenger surname.
    Returns:
        JSON {"verified": bool, "passenger_id": str|null}.
    """
    rec = _PNRS.get(record_locator.upper())
    ok = bool(rec) and rec["surname"].lower() == surname.lower()
    return json.dumps({"verified": ok, "passenger_id": rec["passenger_id"] if ok else None})


@tool
def compute_compensation(disruption_type: str, loyalty_tier: str) -> str:
    """Duty-of-care compensation for a disruption, by type and tier (deterministic policy).

    Args:
        disruption_type: e.g. 'cancellation' or 'delay'.
        loyalty_tier: e.g. 'Gold'.
    Returns:
        JSON with voucher value (INR) and care items.
    """
    base = {"cancellation": 5000, "delay": 2500}.get(disruption_type.lower(), 0)
    bump = {"Gold": 1.5, "Silver": 1.2}.get(loyalty_tier, 1.0)
    care = ["meal voucher", "priority rebooking"]
    if disruption_type.lower() == "cancellation":
        care.append("hotel if overnight")
    return json.dumps({"voucher_inr": int(base * bump), "care": care, "tier_applied": loyalty_tier})


@tool
def apply_policy_gate(record_locator: str, disruption_reason: str) -> str:
    """Deterministic policy gate using ground-truth booking + fare data.

    Args:
        record_locator: PNR.
        disruption_reason: Free-text reason for the change.
    Returns:
        JSON {"pass": bool, "reasons": [...], "requirements": {...}}.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"pass": False, "reasons": ["PNR not found"], "requirements": {}})
    fb = rec["segments"][0]["fare_basis"]
    rules = _FARE_RULES.get(fb, {})
    airline_caused = any(k in disruption_reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    requirements = {"fee_waiver_required": airline_caused, "reaccommodation_required": airline_caused,
                    "refund_promise_allowed": bool(rules.get("refundable")) or airline_caused}
    passed = airline_caused or bool(rules.get("refundable"))
    reasons = [] if passed else ["Voluntary change on a non-refundable fare cannot be auto-approved."]
    return json.dumps({"pass": passed, "reasons": reasons, "requirements": requirements})


@tool
def write_audit(event: str) -> str:
    """Append an immutable audit record for compliance.

    Args:
        event: Description of the decision or action to log.
    Returns:
        JSON confirmation with the audit sequence number.
    """
    seq = len(AUDIT_LOG)
    AUDIT_LOG.append({"seq": seq, "event": event})
    return json.dumps({"logged": True, "seq": seq})

print("Deterministic ops ready: validate_identity, compute_compensation, apply_policy_gate, write_audit")

Deterministic ops ready: validate_identity, compute_compensation, apply_policy_gate, write_audit


## v7: Swarm (autonomous handoffs)

A team of peers with shared memory and no central boss. Any agent can hand control to any other when it hits the edge of its expertise. The path is not scripted; it emerges.

| Concept card | |
|---|---|
| What it is | Peer agents that hand off to each other, sharing context |
| Who controls the path | The model (agents decide handoffs) |
| The move | `Swarm([...])` with handoff-aware prompts and guard rails |
| Cost shape | Variable, potentially high (handoffs are unbounded until you cap them) |

```mermaid
flowchart TD
    RA[reaccom_specialist] <--> FA[fare_specialist]
    RA <--> CA[compensation_specialist]
    FA <--> CA
    RA <--> CO[comms_specialist]
    FA <--> CO
    CA <--> CO
```

**The mechanic:** Strands gives every swarm agent an auto-injected `handoff_to_agent(agent_name, message, context)` tool and a shared context (the task, who worked on it, what they found). The entry agent starts; agents hand off until one produces the final answer.

In [ ]:
from strands.multiagent import Swarm

# Fresh agents for the swarm. Each prompt names WHO to hand to and WHEN.
sw_reaccom = Agent(model=haiku, name="reaccom_specialist",
    system_prompt=("You handle re-accommodation after a disruption. Find alternative flights for the PNR. "
                   "Hand off to fare_specialist if fare rules or fee waivers are unclear. "
                   "Hand off to compensation_specialist if duty-of-care compensation is needed. "
                   "Hand off to comms_specialist once options are ready to be written up."),
    tools=[get_pnr, search_reaccommodation])

sw_fare = Agent(model=haiku, name="fare_specialist",
    system_prompt=("You confirm fare rules and whether the change is airline-caused (involuntary, fees waived). "
                   "Hand off to reaccom_specialist for flight options, compensation_specialist for care, "
                   "or comms_specialist to write the reply."),
    tools=[get_pnr, get_fare_rules])

sw_comp = Agent(model=haiku, name="compensation_specialist",
    system_prompt=("You compute duty-of-care compensation for the disruption type and loyalty tier. "
                   "Hand off to comms_specialist when the package is ready, or back to fare_specialist / "
                   "reaccom_specialist if inputs are missing."),
    tools=[get_pnr, get_loyalty, compute_compensation])

sw_comms = Agent(model=haiku, name="comms_specialist",
    system_prompt=("You write the final warm, correct customer message summarizing the re-accommodation and "
                   "compensation. Do not invent facts; use only what the team gathered. This ends the task."))

irrops_swarm = Swarm(
    [sw_reaccom, sw_fare, sw_comp, sw_comms],
    entry_point=sw_reaccom,
    max_handoffs=12,
    max_iterations=12,
    execution_timeout=600.0,
    node_timeout=180.0,
    repetitive_handoff_detection_window=6,   # guard: look back 6 handoffs
    repetitive_handoff_min_unique_agents=3,  # guard: require 3+ unique agents in that window
)
print("IRROPS swarm built with 4 peers and ping-pong guards.")

IRROPS swarm built with 4 peers and ping-pong guards.


In [8]:
irrops = ("IRROPS: flight 6E-317 BLR-DEL (PNR JX48Q2, passenger Rao, Gold tier) was cancelled by the airline. "
          "Re-accommodate the passenger to BOM today, confirm the change fee is waived because it is involuntary, "
          "compute duty-of-care compensation for a cancellation, and prepare a customer message.")

with meter("v7_swarm_irrops"):
    reply7, res7 = metered_multi(irrops_swarm, irrops)

print("\n----- SWARM FINAL MESSAGE -----\n", reply7)

node=<reaccom_specialist> | node execution failed
Traceback (most recent call last):
  File "/Users/akash-at-work/Documents/IBS Agentic AI and AWS GenAI Training/Day-10 - Strands/.venv/lib/python3.14/site-packages/strands/multiagent/swarm.py", line 790, in _execute_swarm
    async for event in node_stream:
        yield event
  File "/Users/akash-at-work/Documents/IBS Agentic AI and AWS GenAI Training/Day-10 - Strands/.venv/lib/python3.14/site-packages/strands/multiagent/swarm.py", line 467, in _stream_with_timeout
    async with asyncio.timeout(timeout):
               ~~~~~~~~~~~~~~~^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.14/3.14.5/Frameworks/Python.framework/Versions/3.14/lib/python3.14/asyncio/timeouts.py", line 88, in __aenter__
    raise RuntimeError("Timeout should be used inside a task")
RuntimeError: Timeout should be used inside a task


[v7_swarm_irrops]  calls/nodes=1  latency=0.01s  tokens=0in+0out  cost=$0.000000

----- SWARM FINAL MESSAGE -----
 SwarmResult(status=<Status.FAILED: 'failed'>, results={}, accumulated_usage={'inputTokens': 0, 'outputTokens': 0, 'totalTokens': 0}, accumulated_metrics={'latencyMs': 0}, execution_count=0, execution_time=9, interrupts=[], node_history=[])


### Illustrated: the path that emerged

Nobody scripted the order below. The swarm produced it at runtime. Run a different scenario and this path changes.

In [ ]:
import matplotlib.pyplot as plt

path = _nodes_run(res7)
print("emergent handoff path:")
print("   " + "  ->  ".join(path) if path else "   (path unavailable)")

if path:
    xs = list(range(len(path)))
    fig, ax = plt.subplots(figsize=(11, 3.2))
    ax.plot(xs, [1] * len(path), "o-", color="#6a3d9a", linewidth=2, markersize=10)
    for i, name in enumerate(path):
        ax.annotate(name, (i, 1), rotation=25, ha="left", va="bottom", fontsize=9)
    ax.set_yticks([]); ax.set_ylim(0.8, 1.6)
    ax.set_xlabel("handoff step"); ax.set_title("Swarm v7: the path emerged, nobody wrote it")
    ax.margins(x=0.05)
    plt.tight_layout(); plt.show()

### The trap: ping-pong handoffs

Two agents can bounce control forever, each deferring to the other. It burns tokens and never finishes.

```mermaid
flowchart LR
    subgraph Bad [Ping-pong: no progress]
        X[agent A] --> Y[agent B] --> X
    end
    subgraph Good [Healthy: forward progress]
        P[agent A] --> Q[agent B] --> R[agent C] --> Done([answer])
    end
```

Two guards you set on every swarm:
- `repetitive_handoff_detection_window` + `repetitive_handoff_min_unique_agents`: if the last N handoffs did not involve enough distinct agents, the swarm stops the loop.
- `max_handoffs`, `max_iterations`, `execution_timeout`, `node_timeout`: hard ceilings so a stuck swarm cannot run up an open-ended bill.

> **What changes in production**
> - Never ship a swarm without both guard families set. Swarms are the least predictable pattern on cost.
> - Log the full handoff path per case; it is your only window into an emergent flow.
> - Keep swarm scope narrow. The more agents and tools, the more handoff surface and the higher the variance.

## v8: Graph (deterministic, auditable)

The opposite end from a swarm. You define the structure; execution follows it every time. The model still reasons inside nodes, but the flow, the order, and the guardrails are yours. This is how regulated work gets done: predictable path, hard-rule checks, an audit trail.

| Concept card | |
|---|---|
| What it is | A directed graph of nodes (agents + deterministic functions) with dependency edges |
| Who controls the path | You (the structure is fixed; the model works inside nodes) |
| The move | `GraphBuilder`: add nodes, add edges, add conditions, cap executions |
| Cost shape | Predictable (you know every node that can run) |

**The scenario:** an involuntary rebooking that must be auditable. Identity is checked by a hard rule, eligibility and options and compensation are gathered, a policy gate decides pass or fail, and every decision is logged.

```mermaid
flowchart TD
    V[validate identity: hard rule] --> E[eligibility agent]
    E --> RA[reaccom agent]
    E --> CO[compensation agent]
    RA --> G{policy gate: hard rule + audit}
    CO --> G
    G -->|fail, capped| RA
    G -->|pass| F[finalize + audit log]
```

### Design choice worth stating out loud

A graph node can be an agent **or** a pure Python function (a deterministic node with zero model calls). For the hard rules here (identity, policy) we use a temperature-0 agent whose only job is to call one deterministic tool and echo its verdict.

Why not a pure-Python `FunctionNode`? Two honest reasons:
- It runs identically on every Strands version. The custom-node result objects have fields that shift between releases.
- The business logic is still fully deterministic, because it lives inside the tool (plain Python reading ground-truth data), not in model text.

You get determinism where it counts and portability for free. The pure `FunctionNode` pattern is shown as a reference two cells down, for when you want zero model calls in a node.

In [ ]:
from strands.multiagent import GraphBuilder
from strands.multiagent.base import Status

# --- Nodes: two hard-rule gate agents (temperature 0) + three worker agents ---

g_validate = Agent(model=haiku_t0, name="g_validate",
    system_prompt=("Call validate_identity with the record locator and surname from the task. "
                   "Then output EXACTLY one line: 'IDENTITY VERIFIED' or 'IDENTITY DENIED'."),
    tools=[validate_identity])

g_elig = Agent(model=haiku, name="g_elig",
    system_prompt=("Assess this disruption. State clearly whether it is airline-caused (involuntary, so change "
                   "fees are waived) and whether a refund is eligible. Use tools; never invent policy."),
    tools=[get_pnr, get_fare_rules, check_refund_eligibility])

g_reaccom = Agent(model=haiku, name="g_reaccom",
    system_prompt="Find alternative flights to re-accommodate the passenger to their destination. Use tools.",
    tools=[get_pnr, search_reaccommodation])

g_comp = Agent(model=haiku, name="g_comp",
    system_prompt="Compute duty-of-care compensation for the disruption type and loyalty tier. Use tools.",
    tools=[get_pnr, get_loyalty, compute_compensation])

g_gate = Agent(model=haiku_t0, name="g_gate",
    system_prompt=("Call apply_policy_gate with the record locator and disruption reason from the task. "
                   "Call write_audit to record the gate decision (include pass or fail and the reason). "
                   "Then output EXACTLY 'POLICY PASS' or 'POLICY FAIL: <reasons>'."),
    tools=[apply_policy_gate, write_audit])

g_final = Agent(model=haiku_t0, name="g_final",
    system_prompt=("Call write_audit to record that the involuntary rebooking was finalized. Then write the final "
                   "customer message: the new flight, the waived change fee, and the compensation. Warm and concise."),
    tools=[write_audit])
print("v8 nodes ready.")

In [ ]:
# --- Conditions. These read a node's latest result from graph state ---

def identity_ok(state):
    r = state.results.get("validate")
    return bool(r) and "identity verified" in str(r.result).lower()


def policy_passed(state):
    r = state.results.get("gate")
    return bool(r) and "policy pass" in str(r.result).lower()


def policy_failed(state):
    r = state.results.get("gate")
    if not r:
        return False
    t = str(r.result).lower()
    return "policy fail" in t


# AND-condition factory: fire only when ALL required nodes have COMPLETED.
# This is the fix for Python's OR semantics on join nodes (see the next cell).
def all_dependencies_complete(required_nodes):
    def check(state):
        return all(n in state.results and state.results[n].status == Status.COMPLETED
                   for n in required_nodes)
    return check

both_ready = all_dependencies_complete(["reaccom", "comp"])

builder = GraphBuilder()
builder.add_node(g_validate, "validate")
builder.add_node(g_elig,     "eligibility")
builder.add_node(g_reaccom,  "reaccom")
builder.add_node(g_comp,     "comp")
builder.add_node(g_gate,     "gate")
builder.add_node(g_final,    "finalize")

builder.add_edge("validate", "eligibility", condition=identity_ok)   # stop if identity fails
builder.add_edge("eligibility", "reaccom")
builder.add_edge("eligibility", "comp")
builder.add_edge("reaccom", "gate", condition=both_ready)            # AND-join
builder.add_edge("comp",    "gate", condition=both_ready)            # AND-join
builder.add_edge("gate", "reaccom",  condition=policy_failed)        # capped feedback loop
builder.add_edge("gate", "finalize", condition=policy_passed)

builder.set_entry_point("validate")
builder.set_max_node_executions(14)   # hard ceiling (covers the feedback loop)
builder.set_execution_timeout(300)
builder.reset_on_revisit(True)

rebooking_graph = builder.build()
print("Auditable rebooking graph built.")

In [ ]:
# Run it. Clear the audit log first so the printout reflects just this run.
task8 = ("Involuntary rebooking. PNR JX48Q2, passenger surname Rao. Flight 6E-317 BLR-DEL was cancelled by the "
         "airline. Re-accommodate to BOM today. Disruption reason: flight cancelled by airline. Loyalty tier: Gold. "
         "Disruption type: cancellation.")

AUDIT_LOG.clear()
with meter("v8_graph_rebooking"):
    reply8, res8 = metered_multi(rebooking_graph, task8)

print("execution order:", _nodes_run(res8))
print("\nAUDIT LOG (immutable record of the run):")
for e in AUDIT_LOG:
    print(f"   [{e['seq']}] {e['event'][:130]}")
print("\n----- FINAL CUSTOMER MESSAGE -----\n", reply8)

### Illustrated: determinism you can point to

Two things just happened that a swarm cannot give you:
- **`execution order`** is a fixed, inspectable list. Same inputs, same path, every run.
- **The audit log** is a decision-by-decision record. When a regulator or a customer asks "why", you have the answer written down.

That is the whole trade. A swarm explores; a graph testifies.

### Reference: a pure-Python deterministic node

For a node with **zero** model calls, Strands lets you subclass `MultiAgentBase`. This is the canonical shape from the Strands docs. Some result fields are shown as `...` in the docs and vary by SDK version, so fill them per your installed release. We used tool-backed gate agents above so this notebook runs unchanged across versions, but this is the pattern for a genuine non-LLM node:

```python
from strands.multiagent.base import MultiAgentBase, MultiAgentResult, NodeResult, Status
from strands.agent.agent_result import AgentResult
from strands.types.content import ContentBlock, Message

class FunctionNode(MultiAgentBase):
    """Execute a deterministic Python function as a graph node."""
    def __init__(self, func, name):
        super().__init__()
        self.func = func
        self.name = name

    async def invoke_async(self, task, invocation_state=None, **kwargs):
        output = self.func(task if isinstance(task, str) else str(task))
        agent_result = AgentResult(
            stop_reason="end_turn",
            message=Message(role="assistant", content=[ContentBlock(text=str(output))]),
            metrics=None,   # fill per your SDK version
            state={},
        )
        return MultiAgentResult(
            status=Status.COMPLETED,
            results={self.name: NodeResult(result=agent_result)},
        )

# builder.add_node(FunctionNode(validate_identity_fn, "validate"), "validate")
```

Same graph wiring as before; the node just runs Python instead of a model.

### The gotcha that will bite you: OR semantics on joins

In the Python Strands SDK, a node fires when **any one** incoming edge is satisfied. That is OR semantics.

The rebooking graph has a diamond: `gate` depends on both `reaccom` and `comp`. Without a guard, `gate` would fire the moment the **first** of the two finished, and run on half the inputs.

The fix is the `all_dependencies_complete` factory used above:

```python
both_ready = all_dependencies_complete(["reaccom", "comp"])
builder.add_edge("reaccom", "gate", condition=both_ready)
builder.add_edge("comp",    "gate", condition=both_ready)
```

Now `gate` only fires once **both** dependencies report `Status.COMPLETED`.

- Miss this and your aggregator silently runs on partial data. No error, just wrong answers.
- Teach it on day one of graphs. It is the single most common graph bug.

> **What changes in production**
> - Put AND-conditions on every join node. Assume nothing about firing order.
> - Cap cyclic graphs with `set_max_node_executions`; a feedback loop with no ceiling is an unbounded bill.
> - Keep hard rules (identity, policy, limits) in deterministic tools, never in model text.

## v9: Composition (a swarm inside a graph)

The endgame. Nest patterns inside patterns. Put the deterministic shell on the outside for anything regulated, and pocket the open-ended creativity inside a single bounded node.

| Concept card | |
|---|---|
| What it is | A compliance Graph whose "explore options" node is itself a Swarm |
| Who controls the path | You, outside; the model, inside the one creative node |
| The move | `builder.add_node(a_swarm, "options")`: a Swarm is a valid graph node |
| Cost shape | Inherited from each part; bound every sub-pattern |

```mermaid
flowchart TD
    V[validate: hard rule] --> E[eligibility agent]
    E --> SW[[Swarm: explore best package]]
    SW --> G{policy gate: hard rule}
    G -->|pass| F[finalize + audit]
```

The graph supplies rails, order, and audit. The swarm supplies exploratory problem-solving, but only inside one contained step, with its own guards. This is what production systems actually look like.

In [ ]:
# Inner Swarm: a small team that explores the best re-accommodation + compensation package.
c_plan = Agent(model=haiku, name="package_planner",
    system_prompt=("You assemble the best re-accommodation + compensation package for a disruption. "
                   "Hand off to flight_finder for options, care_desk for compensation, and combine their findings."),
    tools=[get_pnr])
c_flight = Agent(model=haiku, name="flight_finder",
    system_prompt="Find alternative flights for the PNR. Hand off to care_desk or package_planner when done.",
    tools=[get_pnr, search_reaccommodation])
c_care = Agent(model=haiku, name="care_desk",
    system_prompt="Compute duty-of-care compensation for the disruption and tier. Hand off to package_planner when done.",
    tools=[get_pnr, get_loyalty, compute_compensation])

options_swarm = Swarm(
    [c_plan, c_flight, c_care],
    entry_point=c_plan,
    max_handoffs=8, max_iterations=8,
    execution_timeout=400.0, node_timeout=150.0,
    repetitive_handoff_detection_window=5,
    repetitive_handoff_min_unique_agents=2,
)

# Outer compliance Graph: hard-rule validate -> eligibility -> [swarm] -> hard-rule gate -> finalize.
v_validate = Agent(model=haiku_t0, name="v_validate",
    system_prompt="Call validate_identity with the record locator and surname. Output EXACTLY 'IDENTITY VERIFIED' or 'IDENTITY DENIED'.",
    tools=[validate_identity])
v_elig = Agent(model=haiku, name="v_elig",
    system_prompt="State whether the disruption is airline-caused (fees waived) and whether a refund is eligible. Use tools.",
    tools=[get_pnr, get_fare_rules, check_refund_eligibility])
v_gate = Agent(model=haiku_t0, name="v_gate",
    system_prompt=("Call apply_policy_gate with the record locator and disruption reason. Call write_audit to log the "
                   "decision. Output EXACTLY 'POLICY PASS' or 'POLICY FAIL: <reasons>'."),
    tools=[apply_policy_gate, write_audit])
v_final = Agent(model=haiku_t0, name="v_final",
    system_prompt="Call write_audit to record finalization. Then write the final customer message. Warm and concise.",
    tools=[write_audit])

cb = GraphBuilder()
cb.add_node(v_validate,    "validate")
cb.add_node(v_elig,        "eligibility")
cb.add_node(options_swarm, "options")     # a Swarm as a single graph node
cb.add_node(v_gate,        "gate")
cb.add_node(v_final,       "finalize")

cb.add_edge("validate", "eligibility", condition=identity_ok)
cb.add_edge("eligibility", "options")
cb.add_edge("options", "gate")
cb.add_edge("gate", "finalize", condition=policy_passed)

cb.set_entry_point("validate")
cb.set_max_node_executions(16)
cb.set_execution_timeout(400)
composed_graph = cb.build()
print("Composed graph (swarm inside graph) built.")

In [ ]:
task9 = ("Involuntary rebooking. PNR JX48Q2, surname Rao. Flight 6E-317 BLR-DEL cancelled by the airline. "
         "Destination BOM today. Disruption reason: flight cancelled by airline. Tier: Gold. Type: cancellation.")

AUDIT_LOG.clear()
with meter("v9_composition"):
    reply9, res9 = metered_multi(composed_graph, task9)

print("outer graph execution order:", _nodes_run(res9))
try:
    inner = res9.results["options"].result
    print("\ninner swarm produced (first 400 chars):\n", str(inner)[:400])
except Exception as e:
    print("\n(inner swarm detail not surfaced in this SDK build)")

print("\nAUDIT LOG:")
for e in AUDIT_LOG:
    print(f"   [{e['seq']}] {e['event'][:120]}")
print("\n----- FINAL MESSAGE -----\n", reply9)

**What composition buys:** the regulator sees a deterministic, audited graph. The passenger gets a package that a creative team of agents actually explored. You did not have to choose between control and creativity; you placed each where it belongs.

Dials are inherited. Bound every sub-pattern (the swarm has its own caps and timeouts) so the whole thing stays predictable.

## Connect all the dots: which pattern, and when

A real ticket lands on your desk. Walk this before you write a line of code.

```mermaid
flowchart TD
    Start([New problem]) --> Q1{Can you draw the full flowchart up front?}
    Q1 -->|Yes| Q2{Does the path change with the input?}
    Q2 -->|No, fixed steps| Chain[Prompt Chaining v2]
    Q2 -->|Yes, by category| Route[Routing v3]
    Q2 -->|Independent subtasks| Parallel[Parallelization v4]
    Q1 -->|No, depends on input| Q3{Is it a known set of specialists?}
    Q3 -->|Yes, delegate at runtime| Orch[Orchestrator-Workers v5]
    Q3 -->|Quality needs iteration| Eval[Evaluator-Optimizer v6]
    Q3 -->|Open-ended collaboration| Swarm[Swarm v7]
    Start --> Q4{Regulated or auditable?}
    Q4 -->|Yes, need a fixed path + audit| Graph[Graph v8]
    Q4 -->|Yes, but one step is creative| Compose[Composition v9]
```

**The default is lower on the ladder, not higher.** Climb only when the rung you are on cannot do the job. Every rung up costs money and predictability.

## The whole day in one table

| Version | Pattern | Who controls path | Determinism | Cost shape | Reach for it when |
|---|---|---|---|---|---|
| v1 | Augmented LLM | Model (one loop) | Medium | Low | Single-domain Q&A with a few tools |
| v2 | Prompt Chaining | You | High | Low, fixed | Fixed multi-step task |
| v3 | Routing | You (fixed logic) | High | Lowest per query | Distinct input categories |
| v4 | Parallelization | You | High | Same or higher | Independent subtasks; high-stakes voting |
| v5 | Orchestrator-Workers | Model (runtime) | Medium | Higher, variable | Messy multi-domain queries |
| v6 | Evaluator-Optimizer | You + model | Medium | Higher per iteration | Clear criteria, iteration helps |
| v7 | Swarm | Model (peers) | Lowest | Variable, high | Open-ended collaboration |
| v8 | Graph | You (structure) | High | Predictable | Regulated / auditable flows |
| v9 | Composition | Mixed | Mixed | Inherited | Regulated shell + creative core |

## Five lines to keep

1. The only question: **who controls the path**, you or the model.
2. Every choice moves three dials: **cost, quality, latency**. Score them before you build.
3. **Start low on the ladder.** Climb only when forced.
4. Workflows for known paths. Agents for unknown ones. Graphs when you need control **and** flexibility. Compose when one step needs freedom inside rails.
5. **Bound everything.** Caps, timeouts, tiered models, AND-conditions on joins. Unbounded agency is an unbounded bill.

You now have all nine patterns running on one airline agent, each measured on the same dials. That is the toolkit. The skill is choosing the smallest one that solves the problem.